<a href="https://colab.research.google.com/github/Fire752000/Stock_screener_select-data-souces/blob/main/%E3%80%8CUntitled1_ipynb%E3%80%8D%E7%9A%84%E5%89%AF%E6%9C%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install yfinance -q

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from datetime import datetime, timedelta

ticker_input = widgets.Text(value='TSLA', description='Ticker:', style={'description_width': '50px'})
years_input = widgets.Dropdown(options=[('1 Year', 1), ('2 Years', 2), ('3 Years', 3), ('5 Years', 5), ('10 Years', 10)], value=5, description='Timeframe:', style={'description_width': '70px'})
refresh_btn = widgets.Button(description='Refresh', button_style='info', layout=widgets.Layout(width='150px', height='35px'))
output_area = widgets.Output()

controls = widgets.HBox([ticker_input, years_input, refresh_btn])
display(controls)
display(output_area)

def run_dashboard(btn):
    with output_area:
        clear_output(wait=True)
        ticker = ticker_input.value.upper()
        years = years_input.value
        print(f"Loading {ticker} data...")
        end_date = datetime.today()
        start_date = end_date - timedelta(days=int(365 * (years + 1)))
        try:
            data = yf.download(ticker, start=start_date, end=end_date)
        except:
            print(f"Error: unable to load {ticker}.")
            return
        if data.empty:
            print(f"Error: no data found for {ticker}.")
            return
        price = data['Close'].squeeze()
        ma200 = price.rolling(window=200).mean()
        rolling_std = price.rolling(window=60).std()
        zscore = (price - ma200) / rolling_std
        slope = (ma200.pct_change(20) * 252 / 20) * 100
        fwd_40d = price.shift(-40) / price - 1
        analysis = pd.DataFrame({'zscore': zscore, 'fwd_40d': fwd_40d}).dropna()
        bins_list = [-np.inf, -2.0, -1.5, -1.0, -0.5, 0.5, 1.0, 1.5, 2.0, np.inf]
        labels = ['< -2.0', '-2.0 to -1.5', '-1.5 to -1.0', '-1.0 to -0.5', '-0.5 to 0.5', '0.5 to 1.0', '1.0 to 1.5', '1.5 to 2.0', '> 2.0']
        analysis['bucket'] = pd.cut(analysis['zscore'], bins=bins_list, labels=labels)
        summary = analysis.groupby('bucket', observed=True)['fwd_40d'].agg(
            Count='count',
            Mean=lambda x: round(x.mean() * 100, 2),
            Median=lambda x: round(x.median() * 100, 2),
            WinRate=lambda x: round((x > 0).mean() * 100, 1)
        )
        summary.columns = ['Count', 'Mean Return %', 'Median Return %', 'Win Rate %']
        cutoff = end_date - timedelta(days=int(365 * years))
        price_plot = price[price.index >= cutoff]
        ma200_plot = ma200[ma200.index >= cutoff]
        zscore_plot = zscore[zscore.index >= cutoff].dropna()
        slope_plot = slope[slope.index >= cutoff].dropna()
        latest_price = price.iloc[-1]
        latest_ma200 = ma200.dropna().iloc[-1]
        latest_zscore = zscore.dropna().iloc[-1]
        latest_slope = slope.dropna().iloc[-1]
        distance = ((latest_price / latest_ma200) - 1) * 100
        current_bucket = "N/A"
        for i in range(len(bins_list) - 1):
            low = bins_list[i]
            high = bins_list[i+1]
            if latest_zscore >= low and latest_zscore < high:
                current_bucket = labels[i]
                break
        if latest_slope > 10:
            regime = "STRONG UPTREND"
        elif latest_slope > 0:
            regime = "WEAK UPTREND"
        elif latest_slope > -10:
            regime = "WEAK DOWNTREND"
        else:
            regime = "STRONG DOWNTREND"
        if latest_zscore < -1.5:
            zsignal = "DEEPLY OVERSOLD"
        elif latest_zscore < -1.0:
            zsignal = "MODERATELY OVERSOLD"
        elif latest_zscore > 1.5:
            zsignal = "DEEPLY OVERBOUGHT"
        else:
            zsignal = "NEUTRAL"
        print(f"\n{'='*60}")
        print(f"  {ticker} Regime Dashboard")
        print(f"{'='*60}")
        print(f"  Price:        ${latest_price:.2f}")
        print(f"  200 DMA:      ${latest_ma200:.2f} ({distance:+.1f}%)")
        print(f"  Z-Score:      {latest_zscore:.2f}  ({zsignal})")
        print(f"  Slope:        {latest_slope:.1f}%  ({regime})")
        print(f"  Current Band: {current_bucket}")
        print(f"{'='*60}\n")
        fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(16, 14), sharex=True, gridspec_kw={'height_ratios': [2.5, 1.2, 1.2]})
        ax1.plot(price_plot.index, price_plot, color='white', linewidth=1.2, label=f'{ticker} Price')
        ax1.plot(ma200_plot.index, ma200_plot, color='#FF6B6B', linewidth=1.5, linestyle='--', label='200 DMA')
        ax1.fill_between(ma200_plot.index, (ma200 - 1.5 * rolling_std).reindex(ma200_plot.index), (ma200 + 1.5 * rolling_std).reindex(ma200_plot.index), alpha=0.12, color='yellow', label='1.5 Sigma Band')
        ax1.set_ylabel('Price ($)', color='white')
        ax1.set_title(f'{ticker} Regime Dashboard ({years}Y)', fontsize=16, fontweight='bold', color='white')
        ax1.legend(loc='upper left', fontsize=9)
        ax1.set_facecolor('#1a1a2e')
        ax1.grid(True, alpha=0.2)
        s_colors = ['#4ECDC4' if s > 0 else '#FF6B6B' for s in slope_plot]
        ax2.bar(slope_plot.index, slope_plot, color=s_colors, alpha=0.7, width=1)
        ax2.axhline(y=0, color='white', linestyle='-', alpha=0.5)
        ax2.set_ylabel('Slope (%)', color='white')
        ax2.set_title('200 DMA Slope (Annualized)', color='white')
        ax2.set_facecolor('#1a1a2e')
        ax2.grid(True, alpha=0.2)
        z_colors = ['#FF6B6B' if z < -1.5 else '#4ECDC4' if z > 1.5 else '#888888' for z in zscore_plot]
        ax3.bar(zscore_plot.index, zscore_plot, color=z_colors, alpha=0.7, width=1)
        ax3.axhline(y=-1.5, color='#FF6B6B', linestyle='--', alpha=0.7, label='Oversold (-1.5)')
        ax3.axhline(y=1.5, color='#4ECDC4', linestyle='--', alpha=0.7, label='Overbought (+1.5)')
        ax3.axhline(y=0, color='white', linestyle='-', alpha=0.3)
        ax3.set_ylabel('Z-Score', color='white')
        ax3.set_title(f'Z-Score vs 200 DMA  (Current: {latest_zscore:.2f})', color='white')
        ax3.set_xlabel('Date', color='white')
        ax3.legend(loc='upper left', fontsize=9)
        ax3.set_facecolor('#1a1a2e')
        ax3.grid(True, alpha=0.2)
        fig.patch.set_facecolor('#0d0d1a')
        for ax in [ax1, ax2, ax3]:
            ax.tick_params(colors='white')
        plt.tight_layout()
        plt.show()
        print(f"\n{'='*60}")
        print(f"  40-Day Forward Returns by Z-Score Band")
        print(f"  Current Band: {current_bucket}")
        print(f"{'='*60}\n")
        for idx_label in labels:
            if idx_label in summary.index:
                row = summary.loc[idx_label]
                marker = " << CURRENT" if idx_label == current_bucket else ""
                print(f"  {idx_label:>15s}  |  Count: {int(row['Count']):>4d}  |  Mean: {row['Mean Return %']:>+7.1f}%  |  Median: {row['Median Return %']:>+7.1f}%  |  Win Rate: {row['Win Rate %']:>5.1f}%{marker}")
        print(f"\n{'='*60}")
        print(f"  Data: {price_plot.index[0].strftime('%Y-%m-%d')} to {price_plot.index[-1].strftime('%Y-%m-%d')} ({len(price_plot)} trading days)")
        print(f"  Shepherd Capital Markets")
        print(f"{'='*60}")

refresh_btn.on_click(run_dashboard)
run_dashboard(None)

Output()